# Spark DataFrame Practical Showcase: Quick-Commerce Orders

This notebook uses a compact quick-commerce dataset so each Spark operation can be taught in isolation.

We will use only three CSV files from the Blinkit-style dataset:

1. `blinkit_orders.csv` - order header information such as order id, customer id, order timestamp, promised delivery time, actual delivery time, delivery status, order total, payment method, delivery partner id, and store id.
2. `blinkit_order_items.csv` - product line items inside each order.
3. `blinkit_products.csv` - product/category attributes such as product name, category, brand, price, MRP, margin percentage, shelf life, and min/max stock levels.

## 1. Session And Path Setup

Run this notebook on EMR with a PySpark kernel. The only value to change for a new class environment is `BUCKET_NAME`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("QuickCommerceSparkShowcase").getOrCreate()

BUCKET_NAME = "ai-de-aws-learning-test"

base_path = f"s3://{BUCKET_NAME}/spark-learning"
raw_path = f"{base_path}/blinkit/raw"
output_path = f"{base_path}/blinkit/output"

orders_csv_path = f"{raw_path}/blinkit_orders.csv"
order_items_csv_path = f"{raw_path}/blinkit_order_items.csv"
products_csv_path = f"{raw_path}/blinkit_products.csv"

print("Raw data path:", raw_path)
print("Output path:", output_path)
print("Spark version:", spark.version)

## 2. Read CSV Without Custom Schema

This first read intentionally avoids a custom schema. It is useful for quick inspection, but it should not be the final pattern for a production pipeline.

In [ ]:
# DataFrame without custom schema.
# Spark reads column names from the header, but it does not receive our business data types.
# Depending on the source values, numeric and date columns may be treated as strings.
orders_without_schema_df = (
    spark.read
    .option("header", True)
    .csv(orders_csv_path)
)

orders_without_schema_df.printSchema()
orders_without_schema_df.show(5, truncate=False)

## 3. Read CSV With Infer Schema And CSV Options

`inferSchema=True` asks Spark to scan data and guess data types. This is convenient for exploration, but it adds extra work during reads and can behave differently if future files contain unexpected values.

In [ ]:
# CSV read with inferSchema and common read options.
# multiLine is useful when a quoted text column can contain line breaks.
# escape handles quotes inside quoted fields.
# mode="PERMISSIVE" keeps rows that Spark can parse and puts malformed fields into nulls where possible.
# For strict pipelines, use mode="FAILFAST" during validation so bad files fail early.
orders_inferred_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(orders_csv_path)
)

orders_inferred_df.printSchema()
orders_inferred_df.show(5, truncate=False)

## 4. Read CSV With Custom Schema

For class consistency, the rest of the notebook uses explicit schemas. If your CSV has extra columns, Spark ignores the extras only when the schema and file are compatible by position; keep the CSV header aligned with these columns for a smooth run.

In [ ]:
# Custom schema for the order header file.
# The schema below matches the sample CSV header exactly:
# order_id, customer_id, order_date, promised_delivery_time, actual_delivery_time,
# delivery_status, order_total, payment_method, delivery_partner_id, store_id.
# Keep timestamp fields as strings during the read, then parse them explicitly in transformation cells.
orders_schema = T.StructType([
    T.StructField("order_id", T.LongType(), False),
    T.StructField("customer_id", T.LongType(), True),
    T.StructField("order_date", T.StringType(), True),
    T.StructField("promised_delivery_time", T.StringType(), True),
    T.StructField("actual_delivery_time", T.StringType(), True),
    T.StructField("delivery_status", T.StringType(), True),
    T.StructField("order_total", T.DoubleType(), True),
    T.StructField("payment_method", T.StringType(), True),
    T.StructField("delivery_partner_id", T.LongType(), True),
    T.StructField("store_id", T.LongType(), True),
])

orders_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("escape", '"')
    .schema(orders_schema)
    .csv(orders_csv_path)
)

orders_df.printSchema()
orders_df.show(5, truncate=False)

## 5. Read The Two Supporting CSV Files

These two DataFrames are enough for realistic joins and product-level analysis.

In [ ]:
order_items_schema = T.StructType([
    T.StructField("order_id", T.LongType(), False),
    T.StructField("product_id", T.LongType(), False),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DoubleType(), True),
])

products_schema = T.StructType([
    T.StructField("product_id", T.LongType(), False),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("brand", T.StringType(), True),
    T.StructField("price", T.DoubleType(), True),
    T.StructField("mrp", T.DoubleType(), True),
    T.StructField("margin_percentage", T.DoubleType(), True),
    T.StructField("shelf_life_days", T.IntegerType(), True),
    T.StructField("min_stock_level", T.IntegerType(), True),
    T.StructField("max_stock_level", T.IntegerType(), True),
])

order_items_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("escape", '"')
    .schema(order_items_schema)
    .csv(order_items_csv_path)
)

products_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("escape", '"')
    .schema(products_schema)
    .csv(products_csv_path)
)

order_items_df.show(5, truncate=False)
products_df.show(5, truncate=False)

## 6. Select

Use `select` when you want a DataFrame with only specific columns.

In [ ]:
# Select can do more than keep column names.
# Here we use Column objects, aliases, and lightweight expressions inside select.
# This is still a select example: no new DataFrame column is being built with withColumn.
orders_selected_df = orders_df.select(
    F.col("order_id").alias("order_key"),
    F.col("customer_id").alias("customer_key"),
    F.col("store_id").alias("dark_store_id"),
    F.col("delivery_status").alias("sla_status"),
    F.col("order_total").alias("gross_order_amount"),
    F.round(F.col("order_total"), 0).cast("long").alias("rounded_order_amount"),
    F.concat_ws("-", F.col("store_id"), F.col("order_id")).alias("store_order_reference")
)

orders_selected_df.show(10, truncate=False)
orders_selected_df.printSchema()

## 7. WithColumn

Use `withColumn` to add a new column or replace an existing column.

In [ ]:
# withColumn is used when the output DataFrame should carry additional derived columns.
# Keep the derivations related: all columns below describe timestamps, SLA timing, or commercial value.
# The source timestamp values look like 2024-07-17 08:34:01, so parse them with the matching pattern.
# Spark property note:
# spark.sql.legacy.timeParserPolicy=LEGACY can relax Spark 3 parsing, but the cleaner fix is to use the correct pattern.
amount_col = F.coalesce(F.col("order_total"), F.lit(0.0))
timestamp_pattern = "yyyy-MM-dd HH:mm:ss"

orders_with_date_df = (
    orders_df
    .withColumn("order_ts", F.to_timestamp(F.col("order_date"), timestamp_pattern))
    .withColumn("promised_delivery_ts", F.to_timestamp(F.col("promised_delivery_time"), timestamp_pattern))
    .withColumn("actual_delivery_ts", F.to_timestamp(F.col("actual_delivery_time"), timestamp_pattern))
    .withColumn("order_dt", F.to_date(F.col("order_ts")))
    .withColumn("order_month", F.date_format(F.col("order_dt"), "yyyy-MM"))
    .withColumn("order_hour", F.hour(F.col("order_ts")))
    .withColumn(
        "delivery_delay_minutes",
        F.round((F.col("actual_delivery_ts").cast("long") - F.col("promised_delivery_ts").cast("long")) / 60, 2)
    )
    .withColumn("platform_fee", F.round(amount_col * F.lit(0.02), 2))
    .withColumn("estimated_seller_payout", F.round(amount_col - F.col("platform_fee"), 2))
)

orders_with_date_df.select(
    "order_id",
    "order_date",
    "order_ts",
    "order_dt",
    "order_month",
    "order_hour",
    "delivery_status",
    "delivery_delay_minutes",
    "order_total",
    "platform_fee",
    "estimated_seller_payout"
).show(10, truncate=False)

# c1 , c2, c3
# 10 , null, 80 - R1
# null, 90, 10 - R2
# null, null, null - R3

# coalesce(c1,c2,c3,0) -> R1 = 10
# coalesce(c1,c2,c3,0) -> R2 = 90
# coalesce(c1,c2,c3,0) -> R3 = 0

## 8. WithColumnRenamed

Use `withColumnRenamed` when a column name is unclear for the next stage.

In [ ]:
# Rename columns when the next DataFrame is meant to be consumed as a cleaner business dataset.
# Chaining is fine when the rename list is short and readable.
orders_renamed_df = (
    orders_df
    .withColumnRenamed("order_id", "order_key")
    .withColumnRenamed("store_id", "dark_store_id")
    .withColumnRenamed("delivery_status", "sla_status")
    .withColumnRenamed("order_total", "gross_order_amount")
)

orders_renamed_df.select(
    "order_key",
    "customer_id",
    "dark_store_id",
    "sla_status",
    "gross_order_amount"
).show(10, truncate=False)

## 9. Filter

Use `filter` to keep rows matching a condition.

In [ ]:
# Build a readable filter using Column objects.
# The condition keeps on-time orders with a usable amount for a focused SLA-quality view.
on_time_orders_df = orders_df.filter(
    (F.col("delivery_status") == F.lit("On Time")) &
    (F.col("order_total").isNotNull()) &
    (F.col("order_total") > F.lit(0)) &
    (F.col("payment_method").isin("UPI", "Card", "Wallet", "Cash"))
)

on_time_orders_df.select(
    "order_id",
    "store_id",
    "delivery_status",
    "payment_method",
    "order_total"
).show(10, truncate=False)

## 10. Drop

Use `drop` to remove columns that are not needed.

In [ ]:
# Drop columns after selecting the grain of the dataset you want to carry forward.
# Here payment_method is removed because this prepared order view is focused on store and SLA performance.
orders_reporting_base_df = (
    orders_df
    .drop("payment_method")
)

orders_reporting_base_df.printSchema()
orders_reporting_base_df.show(5, truncate=False)

## 11. DropDuplicates

Use `dropDuplicates` when a file may contain repeated business keys.

In [ ]:
# Remove duplicate rows by business key.
# A duplicate order_id would inflate order counts and revenue totals.
# * df.dropDuplicates() → Deduplicate on all columns (full row comparison).
# * df.dropDuplicates(["col1", "col2"]) → Deduplicate on selected columns only.
# * df.distinct() → Same behavior as df.dropDuplicates() with no arguments.

# This will perform dedup on complete row level
# orders_deduped_df = orders_df.dropDuplicates()
# print(orders_deduped_df.count())


orders_deduped_df = orders_df.dropDuplicates(["order_id"])

duplicate_summary_df = spark.createDataFrame(
    [
        ("before_drop_duplicates", orders_df.count()),
        ("after_drop_duplicates", orders_deduped_df.count())
    ],
    ["stage", "row_count"]
)

duplicate_summary_df.show(truncate=False)

## 12. OrderBy

Use `orderBy` when the final result needs a predictable order.

In [ ]:
# orderBy is commonly used before displaying top records.
# Use explicit sort directions so the intent is clear to readers.
orders_by_amount_df = orders_df.orderBy(
    F.col("order_total").desc_nulls_last(),
    F.col("order_date").asc_nulls_last()
)

orders_by_amount_df.select(
    F.col("order_id"),
    F.col("store_id"),
    F.col("order_date"),
    F.col("delivery_status"),
    F.col("order_total").alias("gross_order_amount")
).show(10, truncate=False)

## 13. Sort

`sort` is an alias for `orderBy`. It is common to see both names in Spark codebases.

In [ ]:
# sort is an alias for orderBy.
# This example sorts a product dimension in a business-friendly browse order.
products_sorted_df = products_df.sort(
    F.col("category").asc_nulls_last(),
    F.col("brand").asc_nulls_last(),
    F.col("product_name").asc_nulls_last()
)

products_sorted_df.select(
    "category",
    "brand",
    "product_name",
    "price",
    "mrp",
    "min_stock_level",
    "max_stock_level"
).show(20, truncate=False)

## 14. GroupBy

Use `groupBy` with aggregation functions to summarize data. This example summarizes store-level order and SLA performance.

In [ ]:
# Store-level order summary with multiple aggregations.
# Each aggregation has an alias so the output columns can be used directly in reports.
store_order_summary_df = (
    orders_df
    .groupBy(F.col("store_id"))
    .agg(
        F.countDistinct(F.col("order_id")).alias("total_orders"),
        F.countDistinct(F.col("customer_id")).alias("unique_customers"),
        F.round(F.sum(F.col("order_total")), 2).alias("total_revenue"),
        F.round(F.avg(F.col("order_total")), 2).alias("avg_order_value"),
        F.max(F.col("order_total")).alias("highest_order_value")
    )
    .orderBy(F.col("total_revenue").desc_nulls_last())
)

print("Number of partitions:", store_order_summary_df.rdd.getNumPartitions())

# If AQE is Disabled .... spark.conf.set("spark.sql.adaptive.enabled", "false")

# After each wide transformation shuffle partitions get created
# By default 200 shuffle partitions will get crated after each wide transformation
# Because default value of property spark.sql.shuffle.partitions is 200
# We can change this property value if we want different number of shuffle partitions


# If AQE is Enabled ( By default it is enabled )

# AQE does run time optimization , so if it is enabled and very tiny
# 200 partitions got created, AQE will automatically call coalesce(1)
# and make 1 partition .. so after running print(store_order_summary_df.rdd.getNumPartitions())
# you will see output as 1

store_order_summary_df.show(20, truncate=False)

## 15. Case When

Use `when(...).otherwise(...)` for readable conditional business rules.

In [ ]:
# Case-when is useful for turning numeric or status values into business labels.
# The rules are written top to bottom, so put the most specific or highest-priority rule first.
orders_value_bucket_df = orders_df.withColumn(
    "order_value_bucket",
    F.when(F.col("order_total").isNull(), "Unknown")
     .when(F.col("order_total") >= F.lit(1000), "High")
     .when(F.col("order_total") >= F.lit(500), "Medium")
     .otherwise("Low")
).withColumn(
    "sla_bucket",
    F.when(F.col("delivery_status") == "On Time", "Healthy SLA")
     .when(F.col("delivery_status") == "Slightly Delayed", "Watch")
     .when(F.col("delivery_status") == "Significantly Delayed", "Action Needed")
     .otherwise("Unknown")
)

orders_value_bucket_df.select(
    "order_id",
    "delivery_status",
    "sla_bucket",
    "order_total",
    "order_value_bucket"
).show(20, truncate=False)

## 16. Window Functions

Window functions calculate values across related rows without collapsing the result like `groupBy` does.

In [ ]:
# Rank products by revenue inside each category.
# This keeps product-level rows while adding window metrics for comparison.
product_sales_df = (
    order_items_df.alias("oi")
    .join(products_df.alias("p"), on=F.col("oi.product_id") == F.col("p.product_id"), how="inner")
    .select(
        F.col("oi.order_id"),
        F.col("oi.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("oi.quantity"),
        F.col("oi.unit_price"),
        (F.col("oi.quantity") * F.col("oi.unit_price")).alias("line_amount")
    )
    .groupBy("category", "product_id", "product_name")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("line_amount"), 2).alias("product_revenue")
    )
)

category_revenue_window = Window.partitionBy("category").orderBy(F.col("product_revenue").desc())

ranked_products_df = (
    product_sales_df
    .withColumn("category_revenue_rank", F.dense_rank().over(category_revenue_window))
    .withColumn(
        "category_revenue_share",
        F.round(
            F.col("product_revenue") / F.sum("product_revenue").over(Window.partitionBy("category")),
            4
        )
    )
)

ranked_products_df.filter(F.col("category_revenue_rank") <= 5).show(50, truncate=False)

## 17. Register DataFrames As Temp Views

Temporary views let you query DataFrames using Spark SQL. They live only in the current Spark session.

In [ ]:
# These views live only for the current Spark session.
# If the notebook kernel restarts, run this cell again before the SQL examples.
orders_df.createOrReplaceTempView("orders")
order_items_df.createOrReplaceTempView("order_items")
products_df.createOrReplaceTempView("products")

# Verify temp views by querying them directly. This avoids listing the Glue catalog.
spark.sql("""
SELECT 'orders' AS view_name, COUNT(*) AS row_count FROM orders
UNION ALL
SELECT 'order_items' AS view_name, COUNT(*) AS row_count FROM order_items
UNION ALL
SELECT 'products' AS view_name, COUNT(*) AS row_count FROM products
""").show(truncate=False)

## 18. Spark SQL: Basic Query

This query mirrors the earlier DataFrame `groupBy` example using SQL syntax.

In [ ]:
store_sql_summary_df = spark.sql("""
SELECT
    store_id,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(SUM(order_total), 2) AS total_revenue,
    ROUND(AVG(order_total), 2) AS avg_order_value,
    SUM(CASE WHEN delivery_status = 'On Time' THEN 1 ELSE 0 END) AS on_time_orders
FROM orders
WHERE order_total IS NOT NULL
GROUP BY store_id
ORDER BY total_revenue DESC
""")

store_sql_summary_df.show(20, truncate=False)

## 19. Spark SQL: Practical Business Query

This query combines orders, order items, and products to find top revenue categories by store.

In [ ]:
category_store_revenue_df = spark.sql("""
WITH order_items_with_product AS (
    SELECT
        o.store_id,
        o.order_id,
        o.delivery_status,
        oi.product_id,
        oi.quantity,
        oi.unit_price,
        oi.quantity * oi.unit_price AS line_amount
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
),
category_sales AS (
    SELECT
        oip.store_id,
        p.category,
        COUNT(DISTINCT oip.order_id) AS orders,
        SUM(oip.quantity) AS units_sold,
        ROUND(SUM(oip.line_amount), 2) AS revenue,
        SUM(CASE WHEN oip.delivery_status = 'On Time' THEN 1 ELSE 0 END) AS on_time_order_lines
    FROM order_items_with_product oip
    JOIN products p
        ON oip.product_id = p.product_id
    GROUP BY oip.store_id, p.category
)
SELECT
    store_id,
    category,
    orders,
    units_sold,
    revenue,
    on_time_order_lines,
    DENSE_RANK() OVER (PARTITION BY store_id ORDER BY revenue DESC) AS category_rank_in_store
FROM category_sales
ORDER BY store_id, category_rank_in_store
""")

category_store_revenue_df.show(30, truncate=False)

## 20. Broadcast Join

Broadcast joins are useful when one side of the join is small enough to send to every executor.

In [ ]:
# products_df is a dimension-style DataFrame, so it is a good broadcast candidate.
# Spark property note:
# spark.sql.autoBroadcastJoinThreshold controls automatic broadcast size.
# In this example, F.broadcast makes the join strategy explicit for demonstration.
broadcast_join_df = (
    order_items_df.alias("oi")
    .join(F.broadcast(products_df).alias("p"), on=F.col("oi.product_id") == F.col("p.product_id"), how="inner")
    .select(
        F.col("oi.order_id"),
        F.col("oi.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("oi.quantity"),
        F.col("oi.unit_price"),
        (F.col("oi.quantity") * F.col("oi.unit_price")).alias("line_amount")
    )
)

broadcast_join_df.explain("formatted")
broadcast_join_df.show(10, truncate=False)